# 01 Dataset Exploration

## MobileSkinXAI Thesis Project

This notebook performs a publication-oriented exploratory analysis of the two datasets used in the MobileSkinXAI thesis project:

1. **ISIC 2019 Skin Lesion Images for Classification**  
   Used as the primary dataset for training, validation, and testing.

2. **PAD-UFES-20 Skin Cancer Dataset**  
   Used as the external validation dataset to evaluate cross-dataset generalization.

The goal of this notebook is to:

- verify dataset availability,
- analyze class distribution,
- inspect metadata,
- study image properties,
- compare ISIC 2019 and PAD-UFES-20,
- and generate thesis-ready figures and tables.

No model training is performed in this notebook.

## Notebook Outline

1. Dataset Paths
2. Dataset Construction
3. Dataset Verification
4. Class Distribution Analysis
5. Metadata Analysis
6. Image Property Analysis
7. Sample Image Visualization
8. Cross-Dataset Comparison
9. Dataset Summary
10. Conclusions

In [ ]:
import os
import json
import random
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

print("Libraries imported successfully.")

# 1. Dataset Paths

The datasets are loaded directly from Kaggle input directories.

The ISIC 2019 dataset is already organized by diagnostic class folders:

- AK
- BCC
- BKL
- DF
- MEL
- NV
- SCC
- VASC

The PAD-UFES-20 dataset contains images split across three folders:

- imgs_part_1
- imgs_part_2
- imgs_part_3

In [ ]:
# ============================================================
# Kaggle Dataset Paths
# ============================================================

ISIC_ROOT = Path("/kaggle/input/datasets/salviohexia/isic-2019-skin-lesion-images-for-classification")
PAD_ROOT = Path("/kaggle/input/datasets/mahdavi1202/skin-cancer")

PAD_METADATA_PATH = PAD_ROOT / "metadata.csv"

WORKING_DIR = Path("/kaggle/working")
FIGURES_DIR = WORKING_DIR / "figures"
RESULTS_DIR = WORKING_DIR / "results"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("ISIC Root:", ISIC_ROOT)
print("PAD Root:", PAD_ROOT)
print("PAD Metadata:", PAD_METADATA_PATH)
print("Figures will be saved to:", FIGURES_DIR)
print("Results will be saved to:", RESULTS_DIR)

In [ ]:
# ============================================================
# Verify Dataset Folders
# ============================================================

print("ISIC exists:", ISIC_ROOT.exists())
print("PAD exists:", PAD_ROOT.exists())
print("PAD metadata exists:", PAD_METADATA_PATH.exists())

print("\nISIC folders:")
for item in ISIC_ROOT.iterdir():
    if item.is_dir():
        print("-", item.name)

print("\nPAD folders/files:")
for item in PAD_ROOT.iterdir():
    print("-", item.name)

# 2. ISIC 2019 Dataset Construction

Since the Kaggle version of ISIC 2019 is already separated into class folders, we create a dataframe using folder names as labels.

This gives us a clean dataset table with:

- image path,
- image filename,
- class label,
- dataset name.

In [ ]:
# ============================================================
# Build ISIC 2019 DataFrame from Class Folders
# ============================================================

ISIC_CLASSES = ["AK", "BCC", "BKL", "DF", "MEL", "NV", "SCC", "VASC"]

isic_records = []

for label in ISIC_CLASSES:
    class_dir = ISIC_ROOT / label
    
    if not class_dir.exists():
        print(f"Warning: Missing folder {label}")
        continue
    
    image_files = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.jpeg")) + list(class_dir.glob("*.png"))
    
    for img_path in image_files:
        isic_records.append({
            "dataset": "ISIC2019",
            "image_id": img_path.stem,
            "filename": img_path.name,
            "path": str(img_path),
            "label": label
        })

isic_df = pd.DataFrame(isic_records)

print("ISIC 2019 dataframe shape:", isic_df.shape)
display(isic_df.head())

# 3. PAD-UFES-20 Dataset Construction

PAD-UFES-20 images are split into three folders. This section collects all image paths and connects them with metadata when possible.

In [ ]:
# ============================================================
# Load PAD-UFES-20 Metadata
# ============================================================

pad_metadata = pd.read_csv(PAD_METADATA_PATH)

print("PAD-UFES-20 metadata shape:", pad_metadata.shape)
print("\nPAD metadata columns:")
print(pad_metadata.columns.tolist())

display(pad_metadata.head())

In [ ]:
# ============================================================
# Collect PAD-UFES-20 Image Files
# ============================================================

pad_image_files = []

for part in ["imgs_part_1", "imgs_part_2", "imgs_part_3"]:
    part_dir = PAD_ROOT / part / part
    
    if part_dir.exists():
        files = list(part_dir.glob("*.png")) + list(part_dir.glob("*.jpg")) + list(part_dir.glob("*.jpeg"))
        pad_image_files.extend(files)
        print(f"{part}: {len(files)} images")
    else:
        print(f"Warning: {part_dir} not found")

print("\nTotal PAD images found:", len(pad_image_files))
print("Sample PAD image:", pad_image_files[0] if pad_image_files else "No image found")

In [ ]:
# ============================================================
# Detect PAD Image and Label Columns
# ============================================================

possible_image_columns = ["img_id", "image_id", "image", "filename", "file", "name"]
possible_label_columns = ["diagnostic", "diagnosis", "label", "class", "lesion_type"]

pad_image_col = None
pad_label_col = None

for col in possible_image_columns:
    if col in pad_metadata.columns:
        pad_image_col = col
        break

for col in possible_label_columns:
    if col in pad_metadata.columns:
        pad_label_col = col
        break

print("Detected PAD image column:", pad_image_col)
print("Detected PAD label column:", pad_label_col)

if pad_label_col is None:
    print("\nNo automatic label column detected. Please inspect metadata columns above.")

In [ ]:
# ============================================================
# Create PAD Image Path Mapping
# ============================================================

pad_path_map = {}

for path in pad_image_files:
    pad_path_map[path.name] = str(path)
    pad_path_map[path.stem] = str(path)

pad_records = []

if pad_image_col is not None and pad_label_col is not None:
    for _, row in pad_metadata.iterrows():
        image_value = str(row[pad_image_col])
        image_stem = Path(image_value).stem
        
        matched_path = None
        
        if image_value in pad_path_map:
            matched_path = pad_path_map[image_value]
        elif image_stem in pad_path_map:
            matched_path = pad_path_map[image_stem]
        
        pad_records.append({
            "dataset": "PADUFES20",
            "image_id": image_stem,
            "filename": image_value,
            "path": matched_path,
            "label": row[pad_label_col]
        })
else:
    for path in pad_image_files:
        pad_records.append({
            "dataset": "PADUFES20",
            "image_id": path.stem,
            "filename": path.name,
            "path": str(path),
            "label": "unknown"
        })

pad_df = pd.DataFrame(pad_records)

print("PAD-UFES-20 dataframe shape:", pad_df.shape)
display(pad_df.head())

# 4. Dataset Verification

This section verifies:

- total number of images,
- missing image paths,
- duplicate image IDs,
- corrupted image files.

In [ ]:
# ============================================================
# Dataset Verification
# ============================================================

def verify_dataset(df, dataset_name):
    print("=" * 60)
    print(f"Dataset Verification: {dataset_name}")
    print("=" * 60)
    
    total_records = len(df)
    missing_paths = df["path"].isnull().sum()
    duplicate_ids = df["image_id"].duplicated().sum()
    
    print("Total records:", total_records)
    print("Missing image paths:", missing_paths)
    print("Duplicate image IDs:", duplicate_ids)
    
    return {
        "dataset": dataset_name,
        "total_records": total_records,
        "missing_paths": int(missing_paths),
        "duplicate_ids": int(duplicate_ids)
    }

isic_verify = verify_dataset(isic_df, "ISIC2019")
pad_verify = verify_dataset(pad_df, "PADUFES20")

In [ ]:
# ============================================================
# Check Corrupted Images
# ============================================================

def check_corrupted_images(df, dataset_name, max_check=None):
    corrupted = []
    
    valid_df = df.dropna(subset=["path"])
    
    if max_check is not None and len(valid_df) > max_check:
        valid_df = valid_df.sample(max_check, random_state=42)
    
    for _, row in valid_df.iterrows():
        try:
            img = Image.open(row["path"])
            img.verify()
        except Exception as e:
            corrupted.append({
                "image_id": row["image_id"],
                "path": row["path"],
                "error": str(e)
            })
    
    print(f"{dataset_name} corrupted images found:", len(corrupted))
    return pd.DataFrame(corrupted)

isic_corrupted = check_corrupted_images(isic_df, "ISIC2019", max_check=3000)
pad_corrupted = check_corrupted_images(pad_df, "PADUFES20", max_check=None)

isic_corrupted.to_csv(RESULTS_DIR / "isic2019_corrupted_images.csv", index=False)
pad_corrupted.to_csv(RESULTS_DIR / "padufes20_corrupted_images.csv", index=False)

# 5. Class Distribution Analysis

Class distribution analysis is essential because skin lesion datasets are usually imbalanced. Severe imbalance can bias a model toward majority classes such as benign nevi.

In [ ]:
# ============================================================
# Class Distribution Table
# ============================================================

def class_distribution_table(df, dataset_name):
    counts = df["label"].astype(str).value_counts().reset_index()
    counts.columns = ["Class", "Count"]
    counts["Percentage"] = (counts["Count"] / counts["Count"].sum() * 100).round(2)
    counts["Dataset"] = dataset_name
    
    return counts[["Dataset", "Class", "Count", "Percentage"]]

isic_class_dist = class_distribution_table(isic_df, "ISIC2019")
pad_class_dist = class_distribution_table(pad_df, "PADUFES20")

display(isic_class_dist)
display(pad_class_dist)

isic_class_dist.to_csv(RESULTS_DIR / "isic2019_class_distribution.csv", index=False)
pad_class_dist.to_csv(RESULTS_DIR / "padufes20_class_distribution.csv", index=False)

In [ ]:
# ============================================================
# Plot Class Distribution
# ============================================================

def plot_class_distribution(dist_df, dataset_name, save_name):
    plt.figure(figsize=(10, 6))
    plt.bar(dist_df["Class"], dist_df["Count"])
    plt.title(f"{dataset_name} Class Distribution")
    plt.xlabel("Diagnostic Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    save_path = FIGURES_DIR / save_name
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    
    print("Saved:", save_path)

plot_class_distribution(isic_class_dist, "ISIC 2019", "figure_01_isic2019_class_distribution.png")
plot_class_distribution(pad_class_dist, "PAD-UFES-20", "figure_02_padufes20_class_distribution.png")

# 6. Metadata Analysis

PAD-UFES-20 contains useful clinical metadata. ISIC 2019 in this Kaggle version is primarily image-folder based, but the original metadata files are also available.

This section analyzes available metadata fields for thesis documentation.

In [ ]:
# ============================================================
# Load ISIC Metadata Files If Available
# ============================================================

isic_metadata_path = ISIC_ROOT / "ISIC_2019_Training_Metadata.csv"
isic_groundtruth_path = ISIC_ROOT / "ISIC_2019_Training_GroundTruth.csv"

isic_metadata = None
isic_groundtruth = None

if isic_metadata_path.exists():
    isic_metadata = pd.read_csv(isic_metadata_path)
    print("ISIC metadata loaded:", isic_metadata.shape)
    display(isic_metadata.head())

if isic_groundtruth_path.exists():
    isic_groundtruth = pd.read_csv(isic_groundtruth_path)
    print("ISIC ground truth loaded:", isic_groundtruth.shape)
    display(isic_groundtruth.head())

In [ ]:
# ============================================================
# Missing Value Analysis
# ============================================================

def missing_value_table(df, dataset_name):
    missing = df.isnull().sum().reset_index()
    missing.columns = ["Column", "Missing Values"]
    missing["Missing Percentage"] = (missing["Missing Values"] / len(df) * 100).round(2)
    missing = missing.sort_values(by="Missing Percentage", ascending=False)
    
    print(f"\nMissing Value Summary: {dataset_name}")
    display(missing)
    
    return missing

if isic_metadata is not None:
    isic_missing = missing_value_table(isic_metadata, "ISIC2019 Metadata")
    isic_missing.to_csv(RESULTS_DIR / "isic2019_metadata_missing_values.csv", index=False)

pad_missing = missing_value_table(pad_metadata, "PADUFES20 Metadata")
pad_missing.to_csv(RESULTS_DIR / "padufes20_metadata_missing_values.csv", index=False)

In [ ]:
# ============================================================
# Categorical Metadata Visualization
# ============================================================

def plot_metadata_column(df, column, dataset_name):
    if column not in df.columns:
        return
    
    counts = df[column].astype(str).value_counts().head(15)
    
    plt.figure(figsize=(10, 6))
    plt.bar(counts.index, counts.values)
    plt.title(f"{dataset_name}: {column} Distribution")
    plt.xlabel(column)
    plt.ylabel("Count")
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    save_path = FIGURES_DIR / f"{dataset_name.lower()}_{column}_distribution.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    
    print("Saved:", save_path)

if isic_metadata is not None:
    for col in ["sex", "age_approx", "anatom_site_general"]:
        plot_metadata_column(isic_metadata, col, "ISIC2019")

for col in pad_metadata.columns:
    if pad_metadata[col].dtype == "object" or pad_metadata[col].nunique() <= 20:
        plot_metadata_column(pad_metadata, col, "PADUFES20")

# 7. Image Property Analysis

This section calculates image-level statistics:

- width,
- height,
- aspect ratio,
- brightness,
- contrast.

These statistics help justify preprocessing choices such as resizing and normalization.

In [ ]:
# ============================================================
# Image Property Extraction
# ============================================================

def analyze_image_properties(df, dataset_name, sample_limit=None):
    records = []
    
    valid_df = df.dropna(subset=["path"]).copy()
    
    if sample_limit is not None and len(valid_df) > sample_limit:
        valid_df = valid_df.sample(sample_limit, random_state=42)
    
    for _, row in valid_df.iterrows():
        try:
            image = Image.open(row["path"]).convert("RGB")
            arr = np.array(image)
            
            height, width = arr.shape[:2]
            brightness = float(np.mean(arr))
            contrast = float(np.std(arr))
            aspect_ratio = round(width / height, 3)
            
            records.append({
                "dataset": dataset_name,
                "image_id": row["image_id"],
                "label": row["label"],
                "width": width,
                "height": height,
                "aspect_ratio": aspect_ratio,
                "brightness": brightness,
                "contrast": contrast
            })
        
        except Exception as e:
            records.append({
                "dataset": dataset_name,
                "image_id": row["image_id"],
                "label": row["label"],
                "error": str(e)
            })
    
    return pd.DataFrame(records)

isic_image_stats = analyze_image_properties(isic_df, "ISIC2019", sample_limit=4000)
pad_image_stats = analyze_image_properties(pad_df, "PADUFES20", sample_limit=None)

display(isic_image_stats.head())
display(pad_image_stats.head())

isic_image_stats.to_csv(RESULTS_DIR / "isic2019_image_statistics.csv", index=False)
pad_image_stats.to_csv(RESULTS_DIR / "padufes20_image_statistics.csv", index=False)

In [ ]:
# ============================================================
# Resolution Distribution Plot
# ============================================================

def plot_resolution_distribution(stats_df, dataset_name, save_name):
    clean_df = stats_df.dropna(subset=["width", "height"])
    
    plt.figure(figsize=(8, 6))
    plt.scatter(clean_df["width"], clean_df["height"], alpha=0.35)
    plt.title(f"{dataset_name}: Image Resolution Distribution")
    plt.xlabel("Width")
    plt.ylabel("Height")
    plt.tight_layout()
    
    save_path = FIGURES_DIR / save_name
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    
    print("Saved:", save_path)

plot_resolution_distribution(isic_image_stats, "ISIC 2019", "figure_03_isic2019_resolution_distribution.png")
plot_resolution_distribution(pad_image_stats, "PAD-UFES-20", "figure_04_padufes20_resolution_distribution.png")

In [ ]:
# ============================================================
# Brightness and Contrast Distribution
# ============================================================

def plot_histogram(stats_df, column, dataset_name, save_name):
    clean_df = stats_df.dropna(subset=[column])
    
    plt.figure(figsize=(10, 6))
    plt.hist(clean_df[column], bins=40)
    plt.title(f"{dataset_name}: {column.capitalize()} Distribution")
    plt.xlabel(column.capitalize())
    plt.ylabel("Frequency")
    plt.tight_layout()
    
    save_path = FIGURES_DIR / save_name
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    
    print("Saved:", save_path)

plot_histogram(isic_image_stats, "brightness", "ISIC 2019", "figure_05_isic2019_brightness_distribution.png")
plot_histogram(isic_image_stats, "contrast", "ISIC 2019", "figure_06_isic2019_contrast_distribution.png")

plot_histogram(pad_image_stats, "brightness", "PAD-UFES-20", "figure_07_padufes20_brightness_distribution.png")
plot_histogram(pad_image_stats, "contrast", "PAD-UFES-20", "figure_08_padufes20_contrast_distribution.png")

# 8. Sample Image Visualization

This section visualizes representative images from each class.

These figures can be used in the thesis dataset description chapter.

In [ ]:
# ============================================================
# Sample Images by Class
# ============================================================

def show_samples_by_class(df, dataset_name, samples_per_class=5):
    labels = sorted(df["label"].astype(str).unique())
    
    for label in labels:
        class_df = df[df["label"].astype(str) == label].dropna(subset=["path"])
        
        if len(class_df) == 0:
            continue
        
        sample_df = class_df.sample(
            min(samples_per_class, len(class_df)),
            random_state=42
        )
        
        plt.figure(figsize=(15, 4))
        
        for i, (_, row) in enumerate(sample_df.iterrows()):
            img = Image.open(row["path"]).convert("RGB")
            
            plt.subplot(1, samples_per_class, i + 1)
            plt.imshow(img)
            plt.title(label)
            plt.axis("off")
        
        plt.suptitle(f"{dataset_name}: Sample Images for Class {label}", fontsize=14)
        plt.tight_layout()
        
        safe_label = str(label).replace("/", "_").replace(" ", "_")
        save_path = FIGURES_DIR / f"{dataset_name.lower()}_sample_{safe_label}.png"
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.show()
        
        print("Saved:", save_path)

show_samples_by_class(isic_df, "ISIC2019", samples_per_class=5)
show_samples_by_class(pad_df, "PADUFES20", samples_per_class=5)

# 9. Cross-Dataset Comparison

The main research value of MobileSkinXAI comes from evaluating generalization across datasets.

ISIC 2019 and PAD-UFES-20 differ in:

- dataset size,
- class distribution,
- image acquisition condition,
- metadata availability,
- and real-world diversity.

This section creates a summary table for the thesis and paper.

In [ ]:
# ============================================================
# Cross-Dataset Summary
# ============================================================

def get_avg_resolution(stats_df):
    clean_df = stats_df.dropna(subset=["width", "height"])
    
    if len(clean_df) == 0:
        return "N/A"
    
    avg_w = int(clean_df["width"].mean())
    avg_h = int(clean_df["height"].mean())
    
    return f"{avg_w} x {avg_h}"

comparison_df = pd.DataFrame({
    "Property": [
        "Dataset Role",
        "Number of Images",
        "Number of Classes",
        "Average Resolution",
        "Metadata Available",
        "Main Use in Study"
    ],
    "ISIC 2019": [
        "Primary dataset",
        len(isic_df),
        isic_df["label"].nunique(),
        get_avg_resolution(isic_image_stats),
        "Yes" if isic_metadata is not None else "Limited",
        "Training, validation, and internal testing"
    ],
    "PAD-UFES-20": [
        "External validation dataset",
        len(pad_df),
        pad_df["label"].nunique(),
        get_avg_resolution(pad_image_stats),
        "Yes",
        "Cross-dataset generalization testing"
    ]
})

display(comparison_df)

comparison_df.to_csv(RESULTS_DIR / "cross_dataset_comparison.csv", index=False)

# 10. Publication-Ready Dataset Summary

The following summary is exported as JSON so that later notebooks can reuse the dataset statistics without recomputing everything.

In [ ]:
# ============================================================
# Export Dataset Summary
# ============================================================

dataset_summary = {
    "ISIC2019": {
        "num_images": int(len(isic_df)),
        "num_classes": int(isic_df["label"].nunique()),
        "classes": sorted(isic_df["label"].astype(str).unique().tolist()),
        "average_resolution": get_avg_resolution(isic_image_stats),
        "class_distribution": isic_class_dist.to_dict(orient="records")
    },
    "PADUFES20": {
        "num_images": int(len(pad_df)),
        "num_classes": int(pad_df["label"].nunique()),
        "classes": sorted(pad_df["label"].astype(str).unique().tolist()),
        "average_resolution": get_avg_resolution(pad_image_stats),
        "class_distribution": pad_class_dist.to_dict(orient="records")
    }
}

summary_path = RESULTS_DIR / "dataset_summary.json"

with open(summary_path, "w") as f:
    json.dump(dataset_summary, f, indent=4)

print("Dataset summary saved to:", summary_path)

# 11. Final Notebook Summary

This notebook completed the dataset exploration stage of MobileSkinXAI.

Generated outputs include:

- class distribution tables,
- image statistics,
- missing value reports,
- corrupted image reports,
- metadata distribution plots,
- sample image figures,
- cross-dataset comparison table,
- and dataset summary JSON.

These outputs will support the dataset description, methodology, and experimental setup sections of the thesis and journal manuscript.

In [ ]:
# ============================================================
# Final Report
# ============================================================

print("=" * 70)
print("MobileSkinXAI Dataset Exploration Completed")
print("=" * 70)

print("\nISIC 2019")
print("Total images:", len(isic_df))
print("Classes:", isic_df["label"].nunique())
print("Average resolution:", get_avg_resolution(isic_image_stats))

print("\nPAD-UFES-20")
print("Total images:", len(pad_df))
print("Classes:", pad_df["label"].nunique())
print("Average resolution:", get_avg_resolution(pad_image_stats))

print("\nGenerated figures:", FIGURES_DIR)
print("Generated results:", RESULTS_DIR)

print("\nNext Step:")
print("02_preprocessing_pipeline.ipynb")

## Key Findings

- ISIC 2019 contains 25,331 images across 8 classes.
- PAD-UFES-20 contains 2,298 images across 6 classes.
- Both datasets exhibit significant class imbalance.
- Cross-dataset label harmonization is required before training.
- Image resolution varies across datasets, supporting a standardized preprocessing pipeline.

These findings motivate the preprocessing decisions implemented in the next notebook.